# **Unified Military Analytics and Comparison Dashboard**

**Project Statement:**

This project develops a fully interactive dashboard suite for analyzing global military power in 2026. Using Python, we scrape metrics from GlobalFirepower.com for 140+ countries to create a data pipeline that emphasizes cross-platform flexibility for tools like Tableau, Power BI, or Streamlit.

I explored GlobalFirepower website to understand what kind of military metrics are available and which of them are meaningful for comparison. After analyzing the structure of the website and the available indicators, I finalized the important metrics.


**Data Scraping:**

These libraries helped me to send requests to the website, extract structured information from HTML content, store the data into CSV format, and handle delays between requests to avoid overwhelming the server

In [1]:
import requests          # To send HTTP request to the website
from bs4 import BeautifulSoup   # To parse HTML content
import csv               # To store extracted data into CSV file
import time              # To add delay between requests
import re                # To extract patterns like Power Index using regex

**Define Country List and Indicators**

In this step, I define the list of country IDs and the selected military indicators to scrape.

In [3]:
# Country IDs based on website URL structure
country_ids = [
    "afghanistan", "albania", "algeria", "angola", "argentina", "armenia",
        "australia", "austria", "azerbaijan", "bahrain", "bangladesh", "belarus",
        "belgium", "belize", "benin", "bhutan", "bolivia", "bosnia-herzegovina",
        "botswana", "brazil", "bulgaria", "burundi", "cambodia", "cameroon", "canada",
        "central-african-republic", "chad", "chile", "china", "colombia", "croatia",
        "cuba", "cyprus", "czechia", "democratic-republic-of-the-congo", "denmark",
        "djibouti", "dominican-republic", "ecuador", "egypt", "el-salvador", "eritrea",
        "estonia", "ethiopia", "finland", "france", "gabon", "gambia", "georgia",
        "germany", "ghana", "greece", "guatemala", "guinea", "guinea-bissau",
        "honduras", "hungary", "iceland", "india", "indonesia", "iran", "iraq",
        "ireland", "israel", "italy", "ivory-coast", "japan", "jordan", "kazakhstan",
        "kenya", "kosovo", "kuwait", "kyrgyzstan", "laos", "latvia", "lebanon",
        "lesotho", "liberia", "libya", "lithuania", "luxembourg", "madagascar", "malawi",
        "malaysia", "mali", "mauritania", "mauritius", "mexico", "moldova", "mongolia",
        "montenegro", "morocco", "mozambique", "myanmar", "namibia", "nepal",
        "netherlands", "new-zealand", "nicaragua", "niger", "nigeria", "north-korea",
        "north-macedonia", "norway", "oman", "pakistan", "panama", "paraguay", "peru",
        "philippines", "poland", "portugal", "qatar", "republic-of-the-congo", "romania",
        "russia", "saudi-arabia", "senegal", "serbia", "sierra-leone", "singapore",
        "slovakia", "slovenia", "somalia", "south-africa", "south-korea", "south-sudan",
        "spain", "sri-lanka", "sudan", "suriname", "sweden", "switzerland", "syria",
        "taiwan", "tajikistan", "tanzania", "thailand", "tunisia", "turkiye",
        "turkmenistan", "uganda", "ukraine", "united-arab-emirates",
        "united-kingdom", "united-states-of-america", "uruguay", "uzbekistan",
        "venezuela", "vietnam", "yemen", "zambia", "zimbabwe"
]

# Selected military indicators for dashboard comparison
target_labels = [
    "Available Manpower", "Purchasing Power Parity", "Defense Budget",
        "Active Personnel", "Reserve Personnel", "Air Force Personnel",
        "Army Personnel", "Navy Personnel", "Aircraft Total", "Attack",
        "Fighters", "Transport", "Helicopters", "Special-Mission",
        "Trainers", "Attack Helicopters", "Tanker", "Naval Assets",
        "Aircraft Carriers", "Helicopter Carriers", "Destroyers",
        "Frigates", "Corvettes", "Submarines", "Patrol Vessels",
        "Mine Warfare", "Tanks", "Self-Propelled Artillery",
        "Towed Artillery", "Rocket Artillery", "External Debt"
]

Here, I create the CSV file and define the column structure for storing the scraped data.

In [4]:
# Define the output file name where scraped raw data will be stored
filename = "raw military data.csv"

# Open the file in write mode ('w')
file = open(filename, 'w', newline='', encoding='utf-8')

# Create a CSV writer object using DictWriter
writer = csv.DictWriter(
    file,
    fieldnames=["Country Name", "Power Index"] + target_labels
)

# Write the header row (column names) to the CSV file
writer.writeheader()

471

I loop through each country page, extract the required indicators, and prepare structured data.

In [5]:
# Output file name
filename = "raw military data.csv"

# Open CSV file in write mode
with open(filename, 'w', newline='', encoding='utf-8') as f:

    # Define CSV structure
    writer = csv.DictWriter(
        f,
        fieldnames=["Country Name", "Power Index"] + target_labels
    )
    writer.writeheader()

    # Loop through all country IDs
    for cid in country_ids:
        print(f"Fetching data for: {cid}...")

        # Build URL
        url = f"https://www.globalfirepower.com/country-military-strength-detail.php?country_id={cid}"
        headers = {"User-Agent": "Mozilla/5.0"}

        try:
            # Send request and parse page
            response = requests.get(url, headers=headers)
            soup = BeautifulSoup(response.content, "html.parser")
            page_text = response.text

            # Extract Power Index
            power_index = "N/A"
            pwr_match = re.search(r'PwrIndx.*?(\d\.\d{4})', page_text)
            if pwr_match:
                power_index = pwr_match.group(1)

            # Extract Country Name
            country_tag = soup.find('span', class_='textRed')
            country_name = country_tag.text.strip() if country_tag else cid.capitalize()

            # Extract Naval Assets (special handling)
            naval_assets = "N/A"
            naval_fix = re.search(
                r'(Total Assets|Naval Assets).*?(\d[\d,]*)',
                page_text,
                re.I | re.S
            )
            if naval_fix:
                naval_assets = naval_fix.group(2).replace(',', '')

            # Extract standard metric containers
            containers = soup.find_all('div', class_='specsGenContainers')
            scraped_dict = {"Naval Assets": naval_assets}

            for box in containers:
                tag = box.select_one('span.textLarge.textBold.textShadow')
                value_tag = box.select('span.textWhite.textShadow')

                if tag and value_tag:
                    name = tag.text.strip().replace(':', '')
                    val = (
                        value_tag[-1]
                        .text.strip()
                        .replace('Stock:', '')
                        .split('\n')[0]
                        .strip()
                    )
                    scraped_dict[name] = val

            # Prepare row data
            row_data = {
                "Country Name": country_name,
                "Power Index": power_index
            }

            # Map extracted values to target labels
            for label in target_labels:
                if label == "Naval Assets":
                    row_data[label] = scraped_dict.get("Naval Assets", "N/A")
                else:
                    match = next(
                        (v for k, v in scraped_dict.items()
                         if label.lower() in k.lower()),
                        "N/A"
                    )
                    row_data[label] = match

            # Write row to CSV
            writer.writerow(row_data)
            print(f"Saved {country_name}")

            # Polite delay between requests
            time.sleep(1)

        except Exception as e:
            print(f"Error with {cid}: {e}")

print(f"\nSuccess! Data saved to {filename}")

Fetching data for: afghanistan...
Saved Afghanistan
Fetching data for: albania...
Saved Albania
Fetching data for: algeria...
Saved Algeria
Fetching data for: angola...
Saved Angola
Fetching data for: argentina...
Saved Argentina
Fetching data for: armenia...
Saved Armenia
Fetching data for: australia...
Saved Australia
Fetching data for: austria...
Saved Austria
Fetching data for: azerbaijan...
Saved Azerbaijan
Fetching data for: bahrain...
Saved Bahrain
Fetching data for: bangladesh...
Saved Bangladesh
Fetching data for: belarus...
Saved Belarus
Fetching data for: belgium...
Saved Belgium
Fetching data for: belize...
Saved Belize
Fetching data for: benin...
Saved Benin
Fetching data for: bhutan...
Saved Bhutan
Fetching data for: bolivia...
Saved Bolivia
Fetching data for: bosnia-herzegovina...
Saved Bosnia-herzegovina
Fetching data for: botswana...
Saved Botswana
Fetching data for: brazil...
Saved Brazil
Fetching data for: bulgaria...
Saved Bulgaria
Fetching data for: burundi...
Save

The data for all countries was successfully scraped and saved into "raw military data.csv" with a structured format.

**Data Cleaning & Standardization:**

This section cleans the scraped military dataset by converting values to numeric format, standardizing column names, handling missing values, and saving the final cleaned dataset.

In [6]:
import pandas as pd   # Used to load, manipulate, and save the dataset
import numpy as np    # Used for handling numeric operations and missing values (NaN)
import re             # Used to clean and extract numeric values using regular expressions

Load Data
Load the dataset and check basic information.

In [7]:
# Load the raw scraped dataset
df = pd.read_csv("raw military data.csv")

# Display dataset structure (column names, data types, null counts)
print(df.info())

# Display total number of rows (countries scraped)
print("Row Count:", len(df))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 153 entries, 0 to 152
Data columns (total 33 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Country Name              153 non-null    object 
 1   Power Index               153 non-null    float64
 2   Available Manpower        153 non-null    object 
 3   Purchasing Power Parity   153 non-null    object 
 4   Defense Budget            153 non-null    object 
 5   Active Personnel          153 non-null    object 
 6   Reserve Personnel         153 non-null    object 
 7   Air Force Personnel       153 non-null    object 
 8   Army Personnel            153 non-null    object 
 9   Navy Personnel            153 non-null    object 
 10  Aircraft Total            153 non-null    object 
 11  Attack                    153 non-null    object 
 12  Fighters                  153 non-null    object 
 13  Transport                 153 non-null    object 
 14  Helicopter

Clean Numbers
Remove commas and symbols, and convert values to numbers.

In [8]:
# Clean Numeric Columns
def clean_numeric(value):
    if pd.isna(value) or str(value).strip().lower() == "n/a":
        return np.nan

    # Remove commas and dollar signs
    value = re.sub(r"[,$]", "", str(value).strip())

    # Extract numeric part
    match = re.search(r"^(\d+\.?\d*)", value)

    return float(match.group(1)) if match else np.nan


# Apply cleaning to all columns except Country Name
for col in df.columns:
    if col != "Country Name":
        df[col] = df[col].apply(clean_numeric)

Fix Column Names
Make column names clean and consistent.

In [9]:
# Clean and standardize column names
df.columns = (
    df.columns
    .str.strip()                          # Remove leading/trailing spaces
    .str.lower()                          # Convert to lowercase
    .str.replace(" ", "_")                # Replace spaces with underscore
    .str.replace("-", "_")                # Replace hyphens with underscore
    .str.replace(r"[^\w]", "", regex=True) # Remove special characters
)

Handle Missing Values and Save
Replace missing values and save the cleaned dataset.

In [10]:
df.fillna(0, inplace=True)

df.to_csv("military_cleaned_data.csv", index=False)

print("Data saved successfully.")

Data saved successfully.


The dataset has been successfully cleaned, standardized, and saved. All numeric values were converted properly, column names were formatted consistently, and missing values were handled to ensure the dataset is ready for analysis.